# Phase 23 — economic interpretation of multiscale regimes

This notebook builds state cards for the model selected by the frozen phase-22 validation rule. Economic interpretation means describing current hierarchy, persistence, activity, and later diagnostic behaviour. It does not create entries, exits, position direction, or a backtest.

In [1]:
import os

# Reproducible HMM fits (M15): fix the BLAS/OpenMP and PyTorch thread counts before the numerical
# libraries load. A concurrent or differently threaded run can change floating-point reduction order
# and therefore the EM path. Restart the kernel after changing HMM_THREADS.
HMM_THREADS = 1
for _thread_variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_thread_variable, str(HMM_THREADS))

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from dc_basics.data import resolve_data_root
from dc_multiscale_research import (
    FeatureTier,
    ModelFamily,
    ObservationClock,
    REGIME_ORDER,
    add_future_diagnostics,
    build_multiscale_state,
    load_close,
    plot_price_states,
    prepare_data,
    ratio_ladder,
    select_hmm_candidates,
    semantic_state_name,
    transition_matrix,
)

torch.set_num_threads(HMM_THREADS)

DATA_ROOT = resolve_data_root()
SYMBOL = "GBPUSD.r"
THRESHOLDS = ratio_ladder(0.002, 1.5, 5)
CLOCK = ObservationClock.UNION_EVENT
FEATURE_TIER = FeatureTier.DYNAMIC
STATE_COUNTS = (3, 4, 5)
HMM_FAMILIES = (ModelFamily.NORMAL_HMM, ModelFamily.STUDENTT_HMM)
RANDOM_STATE = 17
STABILITY_SEEDS = (29, 47)
MAX_ITER = 50
N_RESTARTS = 3
MINIMUM_VALIDATION_OCCUPANCY = 0.02
MINIMUM_RESTART_ARI = 0.80
MIN_EMISSION_VARIANCE = 1e-2
FUTURE_HORIZONS = (6, 24, 96)
LAST_N = 8_000

## Reproduce the frozen validation selection

No economic profile participates in model selection. The family and state count are chosen by the prefix causality, occupancy, and restart-stability eligibility gates and then by validation likelihood among the eligible candidates; when only one candidate is eligible, the gates alone choose it. The direct structural fields used to interpret a state were not supplied to the continuous emission model.

In [2]:
prices = load_close(DATA_ROOT, SYMBOL)
_, global_state, _ = build_multiscale_state(prices, THRESHOLDS, include_scale_state=False)
prepared = prepare_data(global_state, CLOCK, FEATURE_TIER)

selection = select_hmm_candidates(
    prepared,
    HMM_FAMILIES,
    STATE_COUNTS,
    random_state=RANDOM_STATE,
    stability_seeds=STABILITY_SEEDS,
    max_iter=MAX_ITER,
    n_restarts=N_RESTARTS,
    minimum_validation_occupancy=MINIMUM_VALIDATION_OCCUPANCY,
    minimum_restart_ari=MINIMUM_RESTART_ARI,
    min_emission_variance=MIN_EMISSION_VARIANCE,
)
selection_table = selection.table
selected_fit = selection.selected_fit
if selected_fit is None:
    raise RuntimeError("No eligible HMM candidate; inspect selection_table.")
display(selection_table.set_index(["family", "n_states"]))
display(selection.stability.set_index(["family", "n_states", "seed"]))
print(f"Interpretation model: {selected_fit.family.value}, K={selected_fit.n_states}")

clock     tier  feature_count  \
family       n_states                                             
normal_hmm   3         union_event_time  dynamic             17   
             4         union_event_time  dynamic             17   
             5         union_event_time  dynamic             17   
studentt_hmm 3         union_event_time  dynamic             17   
             4         union_event_time  dynamic             17   
             5         union_event_time  dynamic             17   

                       removed_feature_count  prepared_rows_dropped error  \
family       n_states                                                       
normal_hmm   3                             5                      0  None   
             4                             5                      0  None   
             5                             5                      0  None   
studentt_hmm 3                             5                      0  None   
             4                             5                      0  None   
             5                             5                      0  None   

                       validation_log_likelihood_per_row  \
family       n_states                                      
normal_hmm   3                                -20.035779   
             4                                -18.910471   
             5                                -18.361830   
studentt_hmm 3                                -21.224605   
             4                                -20.268505   
             5                                -19.659651   

                       minimum_state_occupancy  switch_rate  median_dwell  \
family       n_states                                                       
normal_hmm   3                        0.156716     0.453575           1.0   
             4                        0.156716     0.559232           1.0   
             5                        0.101279     0.478122           1.0   
studentt_hmm 3                        0.151386     0.510139           1.0   
             4                        0.148188     0.536820           1.0   
             5                        0.095949     0.436499           1.0   

                       ...  semantic_mapping_coverage  \
family       n_states  ...                              
normal_hmm   3         ...                   1.000000   
             4         ...                   1.000000   
             5         ...                   0.898721   
studentt_hmm 3         ...                   1.000000   
             4         ...                   1.000000   
             5         ...                   0.843284   

                       baseline_mapped_agreement  \
family       n_states                              
normal_hmm   3                          0.666311   
             4                          0.796375   
             5                          0.538553   
studentt_hmm 3                          0.613006   
             4                          0.817697   
             5                          0.628319   

                       training_classes_without_state  \
family       n_states                                   
normal_hmm   3                        (deep_counter,)   
             4                                     ()   
             5                                     ()   
studentt_hmm 3                     (shallow_counter,)   
             4                                     ()   
             5                                     ()   

                       minimum_fitted_variance  variance_floor_hit_count  \
family       n_states                                                      
normal_hmm   3                        0.080025                         0   
             4                        0.065909                         0   
             5                        0.010000                         3   
studentt_hmm 3                        0.075384                         

error  validation_ari_vs_primary  \
family       n_states seed                                    
normal_hmm   3        29    None                   0.281851   
                      47    None                   0.414532   
             4        29    None                   0.427548   
                      47    None                   0.996628   
             5        29    None                   0.269578   
                      47    None                   0.548321   
studentt_hmm 3        29    None                   0.866298   
                      47    None                   0.866298   
             4        29    None                   1.000000   
                      47    None                   0.487473   
             5        29    None                   0.473750   
                      47    None                   0.328640   

                            validation_log_likelihood_per_row  \
family       n_states seed                                      
normal_hmm   3        29                           -20.106149   
                      47                           -20.022606   
             4        29                           -19.019708   
                      47                           -18.909024   
             5        29                           -18.065066   
                      47                           -18.048304   
studentt_hmm 3        29                           -21.284885   
                      47                           -21.284887   
             4        29                           -20.268503   
                      47                           -20.406084   
             5        29                           -19.115560   
                      47                           -19.827081   

                            prefix_invariant  
family       n_states seed                    
normal_hmm   3        29                True  
                      47                True  
             4        29                True  
                      47                True  
             5        29                True  
                      47                True  
studentt_hmm 3        29                True  
                      47                True  
             4        29                True  
                      47                True  
             5        29                True  
                      47                True

Interpretation model: studentt_hmm, K=3


## Construct one causal state path

Raw model states are retained. Training-only semantic names are annotations that explain strongest overlap with the direct taxonomy; they do not redefine the fitted states.

In [3]:
state_parts = []
for split_name in ("train", "validation", "test"):
    part = prepared.frames[split_name].copy()
    part["partition"] = split_name
    part["model_state"] = selected_fit.states[split_name]
    part["model_state_label"] = [f"state_{state}" for state in selected_fit.states[split_name]]
    part["training_overlap_annotation"] = [
        semantic_state_name(code) for code in selected_fit.mapped_regimes[split_name]
    ]
    part["filtered_confidence"] = selected_fit.probabilities[split_name].max(axis=1)
    for state in range(selected_fit.n_states):
        part[f"filtered_p_state_{state}"] = selected_fit.probabilities[split_name][:, state]
    state_parts.append(part)
state_path = pd.concat(state_parts).sort_index()
if not state_path["feature_information_time"].eq(state_path["time"]).all():
    raise AssertionError("A state-path feature timestamp differs from its observation time.")
display(state_path.tail(10))

,time,feature_information_time,bar_position,price,threshold_count,configuration_code,configuration,broad_agreement,coherent_frontier,direction_boundaries,...,log1p_completed_t_04,log_completed_r_04,partition,model_state,model_state_label,training_overlap_annotation,filtered_confidence,filtered_p_state_0,filtered_p_state_1,filtered_p_state_2
time,,,,,,,,,,,,,,,,,,,,,
2026-11-05 14:15:00+02:00,2026-11-05 14:15:00+02:00,2026-11-05 14:15:00+02:00,258005,1.34774,5,29,up/down/up/up/up,False,False,2,...,7.037028,-11.589347,test,1,state_1,fragmented,0.999998,1.773899e-08,9.999981e-01,1.900007e-06
2026-11-05 21:15:00+02:00,2026-11-05 21:15:00+02:00,2026-11-05 21:15:00+02:00,258089,1.34549,5,28,down/down/up/up/up,False,True,1,...,7.037028,-11.589347,test,1,state_1,fragmented,0.999995,7.557990e-12,9.999952e-01,4.776883e-06
2026-11-06 05:15:00+02:00,2026-11-06 05:15:00+02:00,2026-11-06 05:15:00+02:00,258185,1.34374,5,24,down/down/down/up/up,False,True,1,...,7.037028,-11.589347,test,1,state_1,fragmented,0.580960,7.329107e-10,5.809596e-01,4.190404e-01
2026-11-06 09:00:00+02:00,2026-11-06 09:00:00+02:00,2026-11-06 09:00:00+02:00,258230,1.34022,5,16,down/down/down/down/up,False,True,1,...,7.037028,-11.589347,test,2,state_2,deep_counter,0.997263,1.661099e-05,2.720837e-03,9.972626e-01
2026-11-06 15:25:00+02:00,2026-11-06 15:25:00+02:00,2026-11-06 15:25:00+02:00,258307,1.33636,5,0,down/down/down/down/down,True,True,0,...,7.320527,-11.386649,test,0,state_0,broad_coherence,1.000000,1.000000e+00,3.266921e-10,1.072254e-09
2026-11-06 17:20:00+02:00,2026-11-06 17:20:00+02:00,2026-11-06 17:20:00+02:00,258330,1.33768,5,1,up/down/down/down/down,False,True,1,...,7.320527,-11.386649,test,1,state_1,fragmented,0.999913,1.085845e-11,9.999129e-01,8.713942e-05
2026-11-06 21:45:00+02:00,2026-11-06 21:45:00+02:00,2026-11-06 21:45:00+02:00,258383,1.33490,5,0,down/down/down/down/down,True,True,0,...,7.320527,-11.386649,test,0,state_0,broad_coherence,0.999999,9.999995e-01,5.476383e-07,4.657483e-10
2026-11-09 03:15:00+02:00,2026-11-09 03:15:00+02:00,2026-11-09 03:15:00+02:00,258449,1.33713,5,1,up/down/down/down/down,False,True,1,...,7.320527,-11.386649,test,1,state_1,fragmented,0.999992,3.112864e-12,9.999922e-01,7.795872e-06
2026-11-09 08:10:00+02:00,2026-11-09 08:10:00+02:00,2026-11-09 08:10:00+02:00,258508,1.33851,5,3,up/up/down/down/down,False,True,1,...,7.320527,-11.386649,test,1,state_1,fragmented,0.999706,1.285861e-10,9.997061e-01,2.938780e-04


## State cards: current economic structure

These fields answer what the state means **now**: direction balance, hierarchy coherence, scale reach, live reversal pressure, event activity, confidence, and direct-regime overlap.

`counter_depth_fraction` exists only for rows with zero or one boundary; fragmented rows have no single counter depth, so that column's `count` is lower than the others in states that contain fragmented rows. Discrete descriptors such as `fragmentation` and `signed_mode_balance` were not model inputs; they describe each state after fitting.

In [4]:
current_fields = [
    "fragmentation",
    "counter_depth_fraction",
    "signed_mode_balance",
    "mean_mode_age_bars",
    "mean_reversal_progress",
    "max_reversal_progress",
    "hierarchy_dcc_rate",
    "filtered_confidence",
]
state_cards = state_path.groupby("model_state", observed=True)[current_fields].agg(
    ["count", "median", "mean"]
)
baseline_overlap = pd.crosstab(
    state_path["model_state"], state_path["structural_regime"], normalize="index"
).reindex(columns=REGIME_ORDER, fill_value=0.0)
direction_overlap = pd.crosstab(
    state_path["model_state"], state_path["outer_mode"], normalize="index"
)
display(state_cards)
display(baseline_overlap.style.format("{:.2%}"))
display(direction_overlap.style.format("{:.2%}"))

fragmentation                  counter_depth_fraction         \
                    count median      mean                  count median   
model_state                                                                
0                     769   0.00  0.000000                    769    0.0   
1                    3156   0.25  0.404309                   1618    0.4   
2                    1137   0.25  0.352243                    737    0.6   

                      signed_mode_balance                  mean_mode_age_bars  \
                 mean               count median      mean              count   
model_state                                                                     
0            0.000000                 769    1.0  0.045514                769   
1            0.351669                3156    0.2  0.012928               3156   
2            0.662415                1137   -0.2 -0.005805               1137   

             ... mean_reversal_progress max_reversal_progress            \
             ...                   mean                 count    median   
model_state  ...                                                          
0            ...               0.036464                   769  0.067491   
1            ...               0.339896                  3156  0.711559   
2            ...               0.247887                  1137  0.701297   

                      hierarchy_dcc_rate                      \
                 mean              count    median      mean   
model_state                                                    
0            0.091670                769  0.005000  0.005465   
1            0.732210               3156  0.004167  0.004907   
2            0.724815               1137  0.005833  0.007343   

            filtered_confidence                      
                          count    median      mean  
model_state                                          
0                           769  0.999992  0.986833  
1                          3156  0.999534  0.969883  
2                          1137  0.994740  0.932463  

[3 rows x 24 columns]

structural_regime,broad_coherence,shallow_counter,deep_counter,fragmented
model_state,,,,
0,100.00%,0.00%,0.00%,0.00%
1,0.73%,41.44%,9.09%,48.73%
2,0.53%,5.28%,59.01%,35.18%


outer_mode,down,up
model_state,,
0,47.72%,52.28%
1,46.61%,53.39%
2,45.38%,54.62%


## Persistence and transition meaning

Transitions are estimated on the held-out event-time path. They describe how structural events move between inferred states; they are not price-direction forecasts.

In [5]:
test_states = selected_fit.states["test"]
test_transition = pd.DataFrame(
    transition_matrix(test_states, selected_fit.n_states),
    index=[f"state_{state}" for state in range(selected_fit.n_states)],
    columns=[f"state_{state}" for state in range(selected_fit.n_states)],
)
display(test_transition.style.format("{:.3f}"))

semantic_map = pd.Series({
    state: semantic_state_name(regime_code)
    for state, regime_code in selected_fit.semantic_map.items()
}, name="training_overlap_annotation")
display(semantic_map.to_frame())

,state_0,state_1,state_2
state_0,0.008,0.933,0.058
state_1,0.161,0.649,0.190
state_2,0.169,0.437,0.393


,training_overlap_annotation
0,broad_coherence
1,fragmented
2,deep_counter


## Later diagnostic behaviour

Forward absolute movement, realized volatility, and multiscale DCC count help name a regime economically. They remain hindsight outcomes and never enter the causal state assignment. Results are shown separately for the chronological test partition.

In [6]:
diagnostics = add_future_diagnostics(global_state, FUTURE_HORIZONS)
future_columns = []
for horizon in FUTURE_HORIZONS:
    future_columns.extend([
        f"future_return_{horizon}",
        f"future_abs_return_{horizon}",
        f"future_realized_vol_{horizon}",
        f"future_dcc_count_{horizon}",
    ])
test_cards = state_path.loc[state_path["partition"].eq("test")].join(
    diagnostics[future_columns], how="left"
)
future_profile = test_cards.groupby("model_state", observed=True)[future_columns].agg(["median", "mean"])
display(future_profile)

directional_check = test_cards.groupby("model_state", observed=True)[
    [f"future_return_{horizon}" for horizon in FUTURE_HORIZONS]
].mean()
display(directional_check.style.format("{:.4%}"))
print("Directional means are diagnostic and do not authorize a directional regime or strategy interpretation.")

future_return_6           future_abs_return_6            \
                     median      mean              median      mean   
model_state                                                           
0             -5.268208e-05 -0.000044            0.000405  0.000619   
1              5.278816e-05  0.000029            0.000378  0.000527   
2             -3.477180e-08  0.000007            0.000523  0.000673   

            future_realized_vol_6           future_dcc_count_6            \
                           median      mean             median      mean   
model_state                                                                
0                        0.000689  0.000817                0.0  0.066667   
1                        0.000556  0.000635                0.0  0.126143   
2                        0.000808  0.000949                0.0  0.179348   

            future_return_24            ... future_dcc_count_24            \
                      median      mean  ...              median      mean   
model_state                             ...                                 
0                   0.000263  0.000288  ...                 0.0  0.441667   
1                   0.000015  0.000008  ...                 0.0  0.588665   
2                   0.000209  0.000262  ...                 0.5  0.945652   

            future_return_96           future_abs_return_96            \
                      median      mean               median      mean   
model_state                                                             
0                   0.000328  0.000319             0.001586  0.002261   
1                  -0.000034  0.000066             0.001405  0.001881   
2                   0.000207  0.000450             0.001706  0.002344   

            future_realized_vol_96           future_dcc_count_96            
                            median      mean              median      mean  
model_state                                                                 
0                         0.002661  0.002915                 1.0  2.108333  
1                         0.002351  0.002557                 2.0  2.106227  
2                         0.003061  0.003167                 2.0  2.939891  

[3 rows x 24 columns]

,future_return_6,future_return_24,future_return_96
model_state,,,
0,-0.0044%,0.0288%,0.0319%
1,0.0029%,0.0008%,0.0066%
2,0.0007%,0.0262%,0.0450%


Directional means are diagnostic and do not authorize a directional regime or strategy interpretation.


## Actual price path and filtered confidence

In [7]:
probability_columns = [f"filtered_p_state_{state}" for state in range(selected_fit.n_states)]
plot_price_states(
    state_path,
    "model_state_label",
    f"{SYMBOL}: economically inspected raw causal filtered states",
    probability_columns=probability_columns,
    last_n=LAST_N,
).show()
plot_price_states(
    state_path,
    "directed_regime",
    f"{SYMBOL}: direction-aware direct structural comparator on the same observations",
    last_n=LAST_N,
).show()

# Phase 23 completion gate

For each inferred state, be able to state:

1. Its raw state ID and, separately, its training-only overlap annotation.
2. Its current hierarchy shape and direction balance.
3. Whether it represents agreement, a coherent frontier, or fragmentation.
4. Its scale ages, reversal pressure, and DCC activity.
5. Its filtered confidence, persistence, restart stability, and any variance-floor warning.
6. How it transitions to other states.
7. Which later activity/volatility differences are diagnostic rather than causal inputs.
8. Whether it adds a stable distinction beyond both direct baselines.
9. Why none of this yet specifies a position direction, entry, exit, or expected net return.